# 🎙️ Breeze-TTS-2 Documentary Voice Generation (Kaggle T4 Edition)
Optimized for **Kaggle GPU T4** (Single or T4x2). Runs purely via CLI / Python API.


In [ ]:
# Step 1: System setup & dependencies
import os, subprocess, sys

os.makedirs("/kaggle/working", exist_ok=True)
if not os.path.exists("/content"):
    try:
        os.symlink("/kaggle/working", "/content")
    except Exception:
        pass

subprocess.run(["apt-get", "update", "-qq"], check=False)
subprocess.run(["apt-get", "install", "-y", "-qq", "sox", "libsox-fmt-all", "aria2", "ffmpeg"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=False)

if not os.path.exists("/kaggle/working/breeze-tts"):
    subprocess.run(["git", "clone", "-q", "https://github.com/breezeblue-ai/breeze-tts.git", "/kaggle/working/breeze-tts"], check=True)

subprocess.run(["uv", "pip", "install", "--system", "--quiet", "sox", "pydub", "huggingface_hub", "soundfile", "openai-whisper", "qwen-tts", "einops"], check=False)
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchvision"], check=False)
print("✅ Setup finished successfully!")


In [ ]:
# Step 2: Download Breeze-TTS-2 model checkpoint & apply compatibility patches
import os, sys
from huggingface_hub import snapshot_download

MODEL_DIR = "/kaggle/working/Breeze-TTS-2"
OUTPUT_DIR = "/kaggle/working/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Patch Transformers 5+ compatibility in infer.py
infer_path = "/kaggle/working/breeze-tts/infer.py"
if os.path.exists(infer_path):
    with open(infer_path, "r") as f:
        inf = f.read()

    shim = (
        "import transformers.modeling_utils\n"
        "try:\n"
        "    from transformers.initialization import no_init_weights\n"
        "    transformers.modeling_utils.no_init_weights = no_init_weights\n"
        "except ImportError:\n"
        "    from contextlib import contextmanager\n"
        "    @contextmanager\n"
        "    def _dummy(*a, **k): yield\n"
        "    transformers.modeling_utils.no_init_weights = _dummy\n"
    )

    if "no_init_weights" not in inf:
        lines = inf.splitlines(keepends=True)
        insert_idx = 0
        for idx, line in enumerate(lines):
            if "from __future__ import annotations" in line:
                insert_idx = idx + 1
                break
        lines.insert(insert_idx, "\n" + shim + "\n")
        with open(infer_path, "w") as f:
            f.writelines(lines)
        print("✅ Compatibility patch applied.")

# Download Breeze-TTS-2 weights (~7.7 GB)
print("📥 Downloading Breeze-TTS-2 weights (~7.7 GB)... This takes ~1-2 minutes on Kaggle.")
snapshot_download(
    repo_id="BreezeBlue/Breeze-TTS-2",
    local_dir=MODEL_DIR,
    local_dir_use_symlinks=False
)
print("✅ Breeze-TTS-2 Model checkpoint ready!")


In [ ]:
# Step 3: Generate Voice Profile 1 (Classic Historical Narrator)
import os, sys, subprocess

breeze_repo = "/kaggle/working/breeze-tts"
model_dir = "/kaggle/working/Breeze-TTS-2"
output_wav_1 = "/kaggle/working/outputs/apollo_classic_historian.wav"

VOICE_INSTRUCTION_1 = (
    "A mature, deep, resonant male baritone with calm, authoritative pacing, "
    "steady breath control, warm acoustic presence, and cinematic gravitas, "
    "speaking like a seasoned historical documentary narrator."
)

DOCUMENTARY_TEXT = (
    "On July 16, 1969, three men sat atop a three-hundred-and-sixty-three-foot tower of steel and liquid fire. "
    "Behind them were four hundred thousand engineers, technicians, and seamstresses who had built the machine "
    "that would carry humanity to the surface of the Moon. "
    "This was not merely a rocket. It was the greatest technological gamble in human history."
)

cmd_1 = [
    sys.executable, "infer.py", model_dir,
    "--text", DOCUMENTARY_TEXT,
    "--instruction", VOICE_INSTRUCTION_1,
    "--cfg-scale", "4.5",
    "--seed", "42",
    "--output", output_wav_1,
]

print("🚀 Executing Voice Profile 1 synthesis...")
proc_1 = subprocess.run(cmd_1, cwd=breeze_repo, capture_output=True, text=True)
print(proc_1.stdout)
if proc_1.stderr:
    print("[STDERR]", proc_1.stderr)

if os.path.exists(output_wav_1):
    file_size = os.path.getsize(output_wav_1)
    print(f"\n🎉 Success! Audio saved to: {output_wav_1} ({file_size / 1024:.1f} KB)")
else:
    raise RuntimeError("Audio generation for Profile 1 failed.")


In [ ]:
# Step 4: Generate Voice Profile 2 (Cosmic Wonder)
import os, sys, subprocess

breeze_repo = "/kaggle/working/breeze-tts"
model_dir = "/kaggle/working/Breeze-TTS-2"
output_wav_2 = "/kaggle/working/outputs/apollo_cosmic_wonder.wav"

VOICE_INSTRUCTION_2 = (
    "A warm, articulate, middle-aged male voice with an engaging, inquisitive cadence, "
    "clear enunciation, thoughtful pauses, and a subtle sense of wonder and reverence, "
    "ideal for a science and space exploration documentary."
)

DOCUMENTARY_TEXT = (
    "On July 16, 1969, three men sat atop a three-hundred-and-sixty-three-foot tower of steel and liquid fire. "
    "Behind them were four hundred thousand engineers, technicians, and seamstresses who had built the machine "
    "that would carry humanity to the surface of the Moon. "
    "This was not merely a rocket. It was the greatest technological gamble in human history."
)

cmd_2 = [
    sys.executable, "infer.py", model_dir,
    "--text", DOCUMENTARY_TEXT,
    "--instruction", VOICE_INSTRUCTION_2,
    "--cfg-scale", "4.5",
    "--seed", "101",
    "--output", output_wav_2,
]

print("🚀 Executing Voice Profile 2 synthesis...")
proc_2 = subprocess.run(cmd_2, cwd=breeze_repo, capture_output=True, text=True)
print(proc_2.stdout)
if proc_2.stderr:
    print("[STDERR]", proc_2.stderr)

if os.path.exists(output_wav_2):
    file_size = os.path.getsize(output_wav_2)
    print(f"\n🎉 Success! Audio saved to: {output_wav_2} ({file_size / 1024:.1f} KB)")
else:
    raise RuntimeError("Audio generation for Profile 2 failed.")

print("\n📁 All generated outputs:")
for f in os.listdir("/kaggle/working/outputs"):
    p = os.path.join("/kaggle/working/outputs", f)
    print(f" - {f} ({os.path.getsize(p) / 1024:.1f} KB)")
